# Train your goldfish on Colab T4 in ~15 minutes

> *the language model that already forgot this sentence*

This notebook reproduces the full GlubLM training run - a 36M-parameter goldfish language model with a 10-second memory - on a free Colab T4 GPU.

**What you'll get at the end:** a `.pt` checkpoint + tokenizer you can chat with, optionally pushed to your own HuggingFace repo.

**Prerequisites:**
- A free Google account (for Colab)
- ~30 minutes wall time (dataset download + 15 min training)
- Optional: a HuggingFace account + token to publish your trained goldfish

Repo: https://github.com/Den-Sec/glublm - License: AGPL-3.0


## Step 1: Pick a T4 GPU

Go to **Runtime > Change runtime type > T4 GPU > Save**, then run the next cell to verify.


In [ ]:
!nvidia-smi


## Step 2: Install `glublm`

Published on PyPI. Pulls PyTorch as a dep but Colab already has it, so this is quick.


In [ ]:
!pip install -q glublm huggingface_hub


## Step 3: Get the dataset

The 60K-sample goldfish dataset lives on HuggingFace. We also grab the pretrained tokenizer (Byte-Level BPE, 5120 vocab).


In [ ]:
from huggingface_hub import hf_hub_download
import json

tok_path = hf_hub_download("DenSec02/glublm-36m", "tokenizer.json")
data_path = hf_hub_download(
    "DenSec02/glublm-60k-ted", "glublm_60k.json", repo_type="dataset"
)

with open(data_path) as f:
    data = json.load(f)

print(f"{len(data):,} samples")
print("\nExample sample:")
print(json.dumps(data[0], indent=2))


## Step 4: Train (~15 minutes on T4)

15 epochs over 60K samples at batch 64, LR 3e-4 with cosine schedule + 5% warmup.

Expected final loss: ~1.14. If you see much higher, check the GPU is actually being used.


In [ ]:
!glublm train \
  --data "$data_path" \
  --out glublm_60k_15ep.pt \
  --tokenizer-out tokenizer_60k.json \
  --epochs 15 \
  --batch-size 64 \
  --lr 3e-4


## Step 5: Chat with your goldfish

Your model literally cannot remember anything past ~96 tokens. That's the whole point. Be nice, it's always meeting you for the first time.


In [ ]:
!glublm chat \
  --ckpt glublm_60k_15ep.pt \
  --tokenizer tokenizer_60k.json \
  --prompt "hello goldfish"


## Optional: Publish your goldfish on HuggingFace

Uncomment the next cell and fill in your HF username. When `login()` prompts, paste a token with `write` scope from https://huggingface.co/settings/tokens.

Note: the pip package ships a converter script - `python -m glublm.export_hf` - if you want the full safetensors + ONNX pipeline. The cell below is the minimal raw `.pt` upload.


In [ ]:
# from huggingface_hub import HfApi, login
# login()  # paste your write-scoped HF token
#
# HF_USERNAME = "your-username-here"
# REPO_ID = f"{HF_USERNAME}/my-goldfish"
#
# api = HfApi()
# api.create_repo(REPO_ID, exist_ok=True)
# api.upload_file(
#     path_or_fileobj="glublm_60k_15ep.pt",
#     path_in_repo="model.pt",
#     repo_id=REPO_ID,
# )
# api.upload_file(
#     path_or_fileobj="tokenizer_60k.json",
#     path_in_repo="tokenizer.json",
#     repo_id=REPO_ID,
# )
# print(f"https://huggingface.co/{REPO_ID}")
